# Figure S5

In [ ]:
from __future__ import annotations

import ast
import re
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np


ROOT = Path.cwd()
FIGURE_DIR = ROOT / 'Figure'
OUTPUT = FIGURE_DIR / 'FigureS5_datamisfit_status.png'

HANNAH_CASES = {
    'GPT': ROOT / 'Hannah_Inversion_GPT',
    'Gemini': ROOT / 'Hannah_Inversion_gemini',
    'Claude': ROOT / 'Hannah_Inversion_claude',
    'Qwen': ROOT / 'Hannah_Inversion_Qwen',
    'GLM': ROOT / 'Hannah_Inversion_GLM',
}
IOWA_GPT = ROOT / 'Iowa_Inversion_GPT'
COLORS = {
    'GPT': '#1f77b4',
    'Gemini': '#ff7f0e',
    'Claude': '#2ca02c',
    'Qwen': '#d62728',
    'GLM': '#9467bd',
}

mpl.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'axes.unicode_minus': False,
    'axes.linewidth': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})


def latest_output_log(inversion_dir: Path) -> Path:
    logs = sorted((inversion_dir / 'iteration_model').glob('Output_*.txt'))
    if not logs:
        raise FileNotFoundError(f'No iteration output log found in {inversion_dir}.')
    return logs[-1]


def read_total_data_misfit(log_path: Path) -> tuple[np.ndarray, np.ndarray]:
    """Return iteration and gravity-plus-magnetic data misfit from one archived log."""
    pattern = re.compile(r"^\s*(\d+)\s+\[.*?\]\s+\S+\s+(\[.*?\])\s+\[")
    iterations, total_data_misfit = [], []
    for line in log_path.read_text(encoding='utf-8').splitlines():
        match = pattern.match(line)
        if match is None:
            continue
        data_terms = ast.literal_eval(match.group(2))
        if len(data_terms) != 2:
            raise ValueError(f'Expected gravity and magnetic data terms in {log_path}: {line}')
        iterations.append(int(match.group(1)))
        total_data_misfit.append(sum(float(value) for value in data_terms))
    if not iterations:
        raise ValueError(f'No parsable data-misfit rows found in {log_path}.')
    return np.asarray(iterations), np.asarray(total_data_misfit)


def style_axis(axis: plt.Axes) -> None:
    axis.set_yscale('log')
    axis.set_xlabel('Iteration', fontsize=10)
    axis.set_ylabel('Data misfit', fontsize=10)
    axis.tick_params(labelsize=8.5, width=0.8, length=3)
    axis.grid(axis='y', which='major', color='#d9d9d9', linewidth=0.55)
    for spine in axis.spines.values():
        spine.set_visible(True)


def plot_data_misfit_status() -> Path:
    FIGURE_DIR.mkdir(parents=True, exist_ok=True)
    fig, axes = plt.subplots(1, 2, figsize=(8.2, 3.35), constrained_layout=True)

    for label, directory in HANNAH_CASES.items():
        log_path = latest_output_log(directory)
        iteration, misfit = read_total_data_misfit(log_path)
        axes[0].plot(iteration, misfit, color=COLORS[label], linewidth=1.6, label=label)
        print(f'Hannah {label}: {log_path.name}; iterations={iteration[0]}–{iteration[-1]}; final data misfit={misfit[-1]:.1f}')

    iowa_log = latest_output_log(IOWA_GPT)
    iowa_iteration, iowa_misfit = read_total_data_misfit(iowa_log)
    axes[1].plot(iowa_iteration, iowa_misfit, color=COLORS['GPT'], linewidth=1.8)
    print(f'Iowa GPT: {iowa_log.name}; iterations={iowa_iteration[0]}–{iowa_iteration[-1]}; final data misfit={iowa_misfit[-1]:.1f}')

    style_axis(axes[0])
    style_axis(axes[1])
    axes[0].set_title('(a) Hannah', loc='left', fontsize=11, fontweight='bold', pad=5)
    axes[1].set_title('(b) Iowa GPT', loc='left', fontsize=11, fontweight='bold', pad=5)
    axes[0].legend(title='Model', fontsize=8.2, title_fontsize=8.5, loc='upper right', frameon=False)

    fig.savefig(OUTPUT, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return OUTPUT


output_path = plot_data_misfit_status()
print(f'Wrote {output_path.relative_to(ROOT)}')
